# Irrigation estimates across fields and calibration runs

Aggregates simulated irrigation from every field of every calibration run
found under `paths.root_output`, using each run's final stage
(`cal_yearround` for year-round runs, `cal_veg` for seasonal ones):

1. area-weighted average irrigation over all fields ("district" total, in mm)
   at several temporal aggregations, with the 5-95% posterior credible band;
2. season totals per field, and which fields count as irrigated under
   different minimum-total thresholds;
3. backscatter goodness-of-fit per field, compared across runs.

Uncertainty comes from the posterior ensemble `01_calibration` saves per
stage (`irrigation_samples_*.nc`, daily irrigation of ~1000 posterior draws).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from wcm_swb import analysis

file_settings = 'configuration_02_analysis_TEMPLATE.json'
options, paths = analysis.load_analysis_config(file_settings)

outputs = analysis.find_outputs(paths['root_output'], options['runs'], options['opt_obs'])
if outputs.empty:
    raise FileNotFoundError(f"No calibration outputs under {paths['root_output']} -- run 01_calibration first.")
finals = analysis.final_stage_outputs(outputs)

shapes = analysis.field_areas(paths['file_shapes'])
missing = sorted(set(finals.field) - set(shapes.index))
if missing:
    raise KeyError(f'No shapefile (area) for fields {missing}; check paths.file_shapes.')

runs = sorted(finals.run.unique())
label = {r: options['run_labels'].get(r, r) for r in runs}
colors = dict(zip(runs, plt.get_cmap('plasma')(np.linspace(0, 0.8, max(len(runs), 2)))))
season = analysis.irrigation_season(options['opt_year'], options['irri_start_doy'], options['irri_end_doy'])

folder_analysis = paths['folder_analysis']
if paths['opt_save_plots']:
    os.makedirs(folder_analysis, exist_ok=True)

print(f'{len(runs)} run(s): {runs}; {finals.field.nunique()} field(s); '
      f'total area {shapes.area.sum() / 1e4:.1f} ha; season {season[0]:%Y-%m-%d} - {season[1]:%Y-%m-%d}')
finals.groupby('run').field.count().rename('n_fields').to_frame()

## Area-weighted irrigation, all fields

For every posterior draw, each field's irrigation [mm] is weighted by its
area and divided by the total area of the fields in that run (fields are
combined draw-by-draw, i.e. treated as independent). Lines are the ensemble
mean of each window total, shaded bands its 5-95% quantiles.

In [ ]:
missing_samples = finals[finals.irrigation_samples.isna()]
if not missing_samples.empty:
    raise FileNotFoundError('No irrigation_samples_*.nc for:\n'
                            + missing_samples[['run', 'field', 'stage']].to_string()
                            + '\nRe-run 01_calibration (older runs did not write this file).')
district = {
    run: analysis.area_weighted_ensemble(dict(zip(g.field, g.irrigation_samples)), shapes.area)
    for run, g in finals.groupby('run')
}

aggregations = options['aggregations']
fig, axes = plt.subplots(1, len(aggregations), figsize=(19 / 2.54, 6 / 2.54), squeeze=False)
for ax, freq in zip(axes[0], aggregations):
    for run in runs:
        agg = analysis.irrigation_quantiles(district[run], freq, *season)
        ax.plot(agg.index, agg['mean'], '-o', color=colors[run], label=label[run], ms=2)
        ax.fill_between(agg.index, agg['q05'], agg['q95'], color=colors[run], alpha=0.15, lw=0)
    ax.set_title(freq)
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Date')
axes[0, 0].set_ylabel('Irrigation [mm]')
axes[0, 0].legend(loc='upper left')
fig.tight_layout()
if paths['opt_save_plots']:
    fig.savefig(os.path.join(folder_analysis, 'irrigation_timeseries' + paths['extension_plot']),
                dpi=300, bbox_inches='tight')
plt.show()

Season totals (with their 5-95% quantiles across the ensemble), and the
average width of the 5-95% band of the window totals at each aggregation.

In [ ]:
rows = []
for run in runs:
    total = analysis.irrigation_quantiles(district[run], 'YS', *season).iloc[0]
    row = {'run': label[run], 'total_mm': total['mean'], 'total_q05_mm': total['q05'], 'total_q95_mm': total['q95']}
    for freq in aggregations:
        agg = analysis.irrigation_quantiles(district[run], freq, *season)
        row[f'mean_band_width_{freq}_mm'] = (agg['q95'] - agg['q05']).mean()
    rows.append(row)
pd.DataFrame(rows).set_index('run').round(2)

## Per-field season totals

In [ ]:
per_field = []
for row in finals.itertuples():
    ensemble = analysis.read_irrigation_samples(row.irrigation_samples)
    tot = analysis.irrigation_quantiles(ensemble, 'YS', *season).iloc[0]
    area = shapes.area[row.field]
    per_field.append({'run': row.run, 'field': row.field, 'area_ha': area / 1e4,
                      'total_mm': tot['mean'], 'total_q05_mm': tot['q05'], 'total_q95_mm': tot['q95'],
                      'volume_m3': tot['mean'] / 1000 * area})
per_field = pd.DataFrame(per_field)
per_field['field'] = pd.Categorical(per_field.field, sorted(per_field.field.unique(), key=analysis.natural_key))
per_field = per_field.sort_values(['run', 'field'])
per_field.pivot(index='field', columns='run', values='total_mm').round(1)

In [ ]:
fields = list(per_field.field.cat.categories)
width = 0.8 / len(runs)
fig, ax = plt.subplots(figsize=(max(4, 0.4 * len(fields) * len(runs)), 3.5))
for i, run in enumerate(runs):
    d = per_field[per_field.run == run].set_index('field').reindex(fields)
    x = np.arange(len(fields)) + (i - (len(runs) - 1) / 2) * width
    ax.bar(x, d.total_mm, width, color=colors[run], label=label[run],
           yerr=[(d.total_mm - d.total_q05_mm).clip(lower=0), (d.total_q95_mm - d.total_mm).clip(lower=0)],
           capsize=2, error_kw={'lw': 0.8})
ax.set_xticks(np.arange(len(fields)), fields, rotation=45 if len(fields) > 8 else 0)
ax.set_ylabel('Season irrigation [mm]')
ax.legend()
if paths['opt_save_plots']:
    fig.savefig(os.path.join(folder_analysis, 'irrigation_per_field' + paths['extension_plot']),
                dpi=300, bbox_inches='tight')
plt.show()

### Irrigated vs. non-irrigated fields

A field counts as irrigated when its simulated (ensemble-mean) season total
reaches the threshold. For each threshold: how many fields, what share of the total
area, and the total simulated volume over the irrigated fields.

In [ ]:
rows = []
for run in runs:
    d = per_field[per_field.run == run]
    for thr in options['irrigated_thresholds_mm']:
        irrigated = d[d.total_mm >= thr]
        rows.append({'run': label[run], 'threshold_mm': thr, 'n_irrigated': len(irrigated),
                     'n_fields': len(d), 'area_irrigated_pct': 100 * irrigated.area_ha.sum() / d.area_ha.sum(),
                     'volume_irrigated_m3': irrigated.volume_m3.sum(),
                     'range_irrigated_mm': (f'{irrigated.total_mm.min():.1f} - {irrigated.total_mm.max():.1f}'
                                            if len(irrigated) else '-'),
                     'non_irrigated': ', '.join(map(str, d[d.total_mm < thr].field))})
pd.DataFrame(rows).set_index(['run', 'threshold_mm']).round(1)

## Backscatter goodness of fit across fields

Simulated vs. observed sigma0 on the calibration timesteps, one point per
field; dotted red lines mark a perfect fit.

In [ ]:
metrics = analysis.sigma0_metrics(finals, options['opt_obs'])
metrics['run_label'] = metrics.run.map(label)
panels = [('r', 'r [-]', 1), ('bias', 'Bias [dB]', 0), ('rmse', 'RMSE [dB]', 0), ('kge', 'KGE [-]', 1)]

fig, axes = plt.subplots(1, len(panels), figsize=(22 / 2.54, 6 / 2.54))
for ax, (col, ylabel, ref) in zip(axes, panels):
    sns.boxplot(data=metrics, x='run_label', y=col, hue='run_label', ax=ax, palette='deep', legend=False)
    sns.stripplot(data=metrics, x='run_label', y=col, ax=ax, color='k', size=3, jitter=.1, alpha=0.5)
    ax.axhline(ref, color='tab:red', ls='dotted', lw=2, alpha=0.7, zorder=-100)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=15)
fig.subplots_adjust(wspace=.5)
if paths['opt_save_plots']:
    fig.savefig(os.path.join(folder_analysis, 'sigma0_metrics' + paths['extension_plot']),
                dpi=300, bbox_inches='tight')
plt.show()

metrics.groupby('run_label')[['r', 'bias', 'rmse', 'ubrmsd', 'kge']].median().round(3)

## Save aggregated results

With `paths.opt_save_plots: true`, also write the tables above as CSV next to
the figures, for use outside this notebook.

In [ ]:
if paths['opt_save_plots']:
    for run in runs:
        analysis.irrigation_quantiles(district[run], '1D').to_csv(
            os.path.join(folder_analysis, f'irrigation_area_weighted_daily_{run}.csv'))
    per_field.to_csv(os.path.join(folder_analysis, 'irrigation_per_field.csv'), index=False)
    metrics.to_csv(os.path.join(folder_analysis, 'sigma0_metrics.csv'), index=False)